# Interim results from a FelixGenome run in progress

Step-2 shards are written as they finish, so a run that is still going — or one
that failed before `Concat` and `Summarize` — already holds usable results. This
notebook reports per-phenotype chromosome coverage, merges whichever phenotypes
are complete, and plots the three arms, so Figure 3's layout can be settled
before the run ends.

Everything is streamed with `gsutil cat`; only the merged TSVs land on the VM.
Merging and shard bookkeeping are done by `scripts/felix_interim_scan.sh` — the
nested-scatter index and the highest-attempt rule live there and are tested, so
this notebook drives it rather than reimplementing it.

**Read the arms differently.** The ordinary arm is usually complete across all 22
chromosomes, so its λGC and Manhattan are the genome-wide numbers. The repeat
arms lag, because `RuVcf`/`BaVcf` gate them — their λGC is computed over a
**chromosome subset** and must be labelled that way, never quoted as
genome-wide.

| Variable | Default | What it does |
| --- | --- | --- |
| `FELIX_RUN_ROOT` | — | `gs://…/FelixGenome/<workflow-id>`; or set `FELIX_SUBMISSION_ID` + `FELIX_WORKFLOW_ID` |
| `FELIX_PHENOTYPES` | `$WORKSPACE_BUCKET/tractor_mix_pilot/selected_phenotypes.txt` | scatter order; the inner shard index is a line number in this file |
| `FELIX_PHENO_COV` | `$WORKSPACE_BUCKET/tractor_mix_pilot/pheno_cov.tsv` | for case/control counts in the calibration table |
| `FELIX_MIN_CHROMS_MAIN` | `22` | merge a phenotype only at this coverage |
| `FELIX_MIN_CHROMS_REPEAT` | `1` | repeat arms are partial by nature; coverage is reported per phenotype |
| `FELIX_INTERIM_DIR` | `felix_interim` | working directory |

Point it at the run that produced the shards. For a rescued run either works —
the cached shards are copies — but the original is guaranteed to hold them all.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

for _d in (Path.cwd() / "scripts", Path.cwd().parent / "scripts", Path.cwd().parent.parent / "scripts"):
    if (_d / "terra_notebook.py").is_file():
        sys.path.insert(0, str(_d.resolve()))
        break
else:
    _bucket = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
    if not _bucket:
        raise FileNotFoundError("scripts/ not found locally and WORKSPACE_BUCKET is unset.")
    _dest = (Path.cwd() / "scripts").resolve()
    _dest.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["gsutil", "-m", "rsync", "-r", f"{_bucket}/scripts/", str(_dest) + "/"])
    sys.path.insert(0, str(_dest))

from terra_notebook import init_notebook
SCRIPTS = init_notebook("felix_interim_scan.sh")


def sh(cmd: str, check: bool = True) -> int:
    """Run a shell command and fail on a non-zero exit.

    get_ipython().system() returns None even when the command fails, so the
    usual `if rc: raise` never fires.
    """
    print(cmd)
    rc = subprocess.run(cmd, shell=True).returncode
    if check and rc:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")
    return rc


WS = os.environ.get("WORKSPACE_BUCKET", "").rstrip("/")
assert WS, "WORKSPACE_BUCKET is unset"

INTERIM = SCRIPTS / "felix_interim_scan.sh"
assert INTERIM.is_file(), INTERIM

# summarize_felix_results.py lives under felix/scripts/ in the bucket, which the
# scripts/ bootstrap does not cover.
SUMMARIZE = Path("summarize_felix_results.py")
if not SUMMARIZE.exists():
    sh(f"gsutil cp {WS}/felix/scripts/summarize_felix_results.py {SUMMARIZE}")
print("interim scan :", INTERIM)
print("summarizer   :", SUMMARIZE.resolve())


In [ ]:
RUN_ROOT = os.environ.get("FELIX_RUN_ROOT", "").rstrip("/")
if not RUN_ROOT:
    sid, wid = os.environ.get("FELIX_SUBMISSION_ID", ""), os.environ.get("FELIX_WORKFLOW_ID", "")
    name = os.environ.get("WORKFLOW_NAME", "FelixGenome")
    if not (sid and wid):
        raise SystemExit(
            "Set FELIX_RUN_ROOT, or FELIX_SUBMISSION_ID + FELIX_WORKFLOW_ID."
        )
    RUN_ROOT = f"{WS}/submissions/{sid}/{name}/{wid}"

PHENOTYPES = os.environ.get("FELIX_PHENOTYPES", f"{WS}/tractor_mix_pilot/selected_phenotypes.txt")
PHENO_COV = os.environ.get("FELIX_PHENO_COV", f"{WS}/tractor_mix_pilot/pheno_cov.tsv")
WORK = Path(os.environ.get("FELIX_INTERIM_DIR", "felix_interim"))
WORK.mkdir(parents=True, exist_ok=True)

MIN_MAIN = int(os.environ.get("FELIX_MIN_CHROMS_MAIN", "22"))
MIN_REPEAT = int(os.environ.get("FELIX_MIN_CHROMS_REPEAT", "1"))
ARMS = {"main": MIN_MAIN, "ru": MIN_REPEAT, "ru_biallelic": MIN_REPEAT}

print("run    :", RUN_ROOT)
print("phenos :", PHENOTYPES)
print("work   :", WORK.resolve())
print("arms   :", ARMS)


In [ ]:
import pandas as pd

cov = {}
for arm in ARMS:
    out = WORK / f"interim_{arm}"
    sh(f"bash {INTERIM} --run {RUN_ROOT} --phenotypes {PHENOTYPES} "
       f"--arm {arm} --min-chroms {ARMS[arm]} --out {out} --coverage-only", check=False)
    f = out / "coverage.tsv"
    if f.is_file():
        cov[arm] = pd.read_csv(f, sep="\t")
    else:
        print(f"  no shards yet for arm {arm}")

if not cov:
    raise SystemExit("No step-2 shards found in any arm yet.")

wide = None
for arm, df in cov.items():
    col = df.set_index("phenotype")["chromosomes_present"].rename(arm)
    wide = col.to_frame() if wide is None else wide.join(col, how="outer")
wide = wide.fillna(0).astype(int)
print("\nchromosomes present per phenotype:")
print(wide.to_string())
print("\nmax coverage per arm:", dict(wide.max()))
print("A repeat-arm number below 22 means that arm's figures are a chromosome subset.")


In [ ]:
merged = {}
for arm, minc in ARMS.items():
    if arm not in cov:
        continue
    out = WORK / f"interim_{arm}"
    # For the repeat arms take every phenotype that has any shard; coverage is
    # reported above and travels with the figures.
    sh(f"bash {INTERIM} --run {RUN_ROOT} --phenotypes {PHENOTYPES} "
       f"--arm {arm} --min-chroms {minc} --out {out}", check=False)
    files = sorted(out.glob("*.merged.felix.tsv"))
    merged[arm] = files
    print(f"{arm}: {len(files)} merged phenotype TSVs, "
          f"{sum(f.stat().st_size for f in files) / 1e9:.2f} GB")


In [ ]:
calib = {}
for arm, files in merged.items():
    if not files:
        continue
    phenos = [f.name.replace(".merged.felix.tsv", "") for f in files]
    sdir = WORK / f"interim_{arm}" / "summary"
    # summarize_felix_results.py takes no --arm; the WDL uses that only to name
    # its bundle directory.
    sh(f"python3 {SUMMARIZE} --results {' '.join(str(f) for f in files)} "
       f"--phenotypes {' '.join(phenos)} --pheno-cov {PHENO_COV} --out-dir {sdir}",
       check=False)
    f = sdir / "calibration_summary.tsv"
    if f.is_file():
        calib[arm] = pd.read_csv(f, sep="\t")

if calib:
    lam = None
    for arm, df in calib.items():
        c = [x for x in df.columns if "lambda" in x.lower()]
        if not c:
            continue
        col = df.set_index("phenotype")[c[0]].rename(arm)
        lam = col.to_frame() if lam is None else lam.join(col, how="outer")
    if lam is not None:
        print("lambda_gc by phenotype and arm "
              "(repeat arms over the chromosome subset shown earlier):")
        print(lam.round(4).to_string())
        lam.to_csv(WORK / "interim_lambda_gc.tsv", sep="\t")
    for arm, df in calib.items():
        keep = [c for c in ("phenotype", "n_variants", "n_cases", "n_controls") if c in df.columns]
        if keep:
            print(f"\n{arm}:")
            print(df[keep].to_string(index=False))


In [ ]:
from IPython.display import Image, display

for arm in ARMS:
    qc = WORK / f"interim_{arm}" / "summary" / "qc"
    if not qc.is_dir():
        continue
    pngs = sorted(qc.glob("*/qq_cct.png")) + sorted(qc.glob("*/manhattan_cct.png"))
    print(f"=== {arm}: {len(pngs)} plots ===")
    for p in pngs[:8]:
        print(f"{arm} / {p.parent.name} / {p.name}")
        display(Image(filename=str(p)))


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(SUMMARIZE.parent.resolve()))
from summarize_felix_results import pick_joint_p_column   # tested column choice


def neglog10(s):
    v = pd.to_numeric(s, errors="coerce").to_numpy(dtype=float)
    v = np.clip(v, 1e-300, 1.0)
    return -np.log10(v)


def read_cols(path, wanted):
    head = pd.read_csv(path, sep="\t", nrows=0)
    use = [c for c in wanted if c in head.columns]
    return pd.read_csv(path, sep="\t", usecols=use, low_memory=False), set(head.columns)


# Panel C analogue: repeat dosage against the carrier-coded comparator, same loci.
pairs = []
for f in merged.get("ru", []):
    g = WORK / "interim_ru_biallelic" / f.name
    if g.is_file():
        pairs.append((f.name.replace(".merged.felix.tsv", ""), f, g))

if pairs:
    rows = []
    for pheno, f, g in pairs:
        a, _ = read_cols(f, ["MarkerID", "P_cct_admixed_c", "P_hom_admixed_c", "p.value_ancALL"])
        b, _ = read_cols(g, ["MarkerID", "P_cct_admixed_c", "P_hom_admixed_c", "p.value_ancALL"])
        pa, pb = pick_joint_p_column(a), pick_joint_p_column(b)
        if not (pa and pb and "MarkerID" in a and "MarkerID" in b):
            continue
        j = a[["MarkerID", pa]].merge(b[["MarkerID", pb]], on="MarkerID",
                                      suffixes=("_dosage", "_carrier"))
        j["phenotype"] = pheno
        rows.append(j.rename(columns={j.columns[1]: "p_dosage", j.columns[2]: "p_carrier"}))
    if rows:
        dvb = pd.concat(rows, ignore_index=True)
        dvb.to_csv(WORK / "interim_dosage_vs_biallelic.tsv.gz", sep="\t", index=False)
        x, y = neglog10(dvb["p_carrier"]), neglog10(dvb["p_dosage"])
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.scatter(x, y, s=3, alpha=0.25, edgecolors="none")
        lim = max(x.max(), y.max()) * 1.05
        ax.plot([0, lim], [0, lim], lw=0.8, color="0.4")
        ax.set_xlabel("-log10 p, carrier coding")
        ax.set_ylabel("-log10 p, repeat dosage")
        ax.set_title(f"dosage vs carrier, {len(pairs)} phenotypes, chromosome subset")
        fig.tight_layout()
        fig.savefig(WORK / "interim_dosage_vs_biallelic.png", dpi=150)
        display(Image(filename=str(WORK / "interim_dosage_vs_biallelic.png")))
        print(f"{len(dvb):,} locus x phenotype points; "
              f"above the diagonal by >1 log unit: {int(((y - x) > 1).sum()):,}; "
              f"below: {int(((x - y) > 1).sum()):,}")
else:
    print("no phenotype has both repeat arms merged yet")


# Panel D analogue: ancestry-aware CCT against FELIX's own pooled pass.
rows = []
for f in merged.get("ru", []):
    a, cols = read_cols(f, ["MarkerID", "P_cct_admixed_c", "p.value_ancALL", "P_het_admixed_c"])
    if "P_cct_admixed_c" in a and "p.value_ancALL" in a:
        a["phenotype"] = f.name.replace(".merged.felix.tsv", "")
        rows.append(a)
    else:
        print(f"  {f.name}: need P_cct_admixed_c and p.value_ancALL; have {sorted(cols)[:8]}...")
if rows:
    avp = pd.concat(rows, ignore_index=True)
    avp.to_csv(WORK / "interim_ancestry_vs_pooled.tsv.gz", sep="\t", index=False)
    x, y = neglog10(avp["p.value_ancALL"]), neglog10(avp["P_cct_admixed_c"])
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(x, y, s=3, alpha=0.25, edgecolors="none")
    lim = max(x.max(), y.max()) * 1.05
    ax.plot([0, lim], [0, lim], lw=0.8, color="0.4")
    ax.set_xlabel("-log10 p, pooled (p.value_ancALL)")
    ax.set_ylabel("-log10 p, ancestry-aware (P_cct_admixed_c)")
    ax.set_title("ancestry-aware vs pooled, chromosome subset")
    fig.tight_layout()
    fig.savefig(WORK / "interim_ancestry_vs_pooled.png", dpi=150)
    display(Image(filename=str(WORK / "interim_ancestry_vs_pooled.png")))
    print(f"{len(avp):,} points; ancestry-aware stronger by >1 log unit: "
          f"{int(((y - x) > 1).sum()):,}; weaker: {int(((x - y) > 1).sum()):,}")


In [ ]:
lines = ["# Interim FelixGenome results", "", f"Run: `{RUN_ROOT}`", ""]
lines += ["## Chromosome coverage per phenotype and arm", "", "```", wide.to_string(), "```", ""]
if "lam" in dir() and lam is not None:
    lines += ["## lambda_gc by arm", "",
              "Repeat-arm values are over the chromosome subset above, not genome-wide.",
              "", "```", lam.round(4).to_string(), "```", ""]
outs = sorted(p.name for p in WORK.glob("interim_*"))
lines += ["## Files", ""] + [f"- `{n}`" for n in outs]
note = WORK / "interim_review.md"
note.write_text("\n".join(lines) + "\n")
print(note)
print("\n".join(lines))
